# Running instructions

In order to run the following test script, you must start a docker container by running `docker start qdrant` and be sure to stop it with `docker stop qdrant`.
The docker container was initialized by running `docker run -p 6333:6333 qdrant_data:/qdrant/storage qdrant/qdrant`.


Feel free to play around with models and data for rag database

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

import os
os.environ["GROQ_API_KEY"] = "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"

# initialize embedding model & vector db
embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
# create or pull collection
# collection = client.create_collection(name="docs")
collection = client.get_collection(name="docs")

# add local text data
texts = ['RAG combines retrieval and generation.', 'embeddings map text to vector space.']
embeddings = embedder.encode(texts).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=['1','2'])

# query
query = 'What does RAG do?'
query_emb = embedder.encode([query]).tolist()
results = collection.query(query_embeddings=query_emb, n_results=1)

context = results['documents'][0][0]

# generate using OpenAI
client = Groq()
prompt = f'Context: {context}\n\nQuestion: {query}\nAnswer:'
response = client.chat.completions.create(model='llama-3.1-8b-instant', messages=[{'role':'user','content':prompt}])
print(response.choices[0].message.content)

RAG (Retrieval-Augmented Generation) combines retrieval and generation to produce a response. This allows a model to retrieve specific information from a large database and use that information to generate a response.
